# 🎯 [C202_M07] 최우선 순위 전문 강제 전송 및 큐 순서 수동 조정을 위한 통신관 HITL 통제 기능 설계
### Google Colab 실습 환경 (지휘통제(C2) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C202/c202_m07_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 2) |
| **미션 ID** | C202_M07 |
| **미션 주제** | 최우선 순위 전문 강제 전송 및 큐 순서 수동 조정을 위한 통신관 HITL 통제 기능 설계 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 전자전 재밍과 통신 장비 부족으로 협대역 무선망의 대역폭이 크게 줄어드는 상황이 잦았고, 이때 적 접촉 보고나 의무후송 요청 같은 긴급 전문이 행정 전문 뒤에 묶여 늦게 전달되는 문제가 지휘통제의 약점이 되었다. 본 프로젝트에서는 가상 협대역 링크(2.4kbps)와 가상 전술 전문을 코드로 직접 만들어, 전문 구조 정의, 패킷 파싱·압축 전처리, 긴급도 기반 우선순위 큐, 채널 상태에 따른 지능형 재전송, 링크 신뢰도 산출, 통신 관제 화면, 통신관 HITL 통제, 대역폭 급감 스트레스 시험, AAR 재조정, 최종 통합까지 메시지 우선순위 큐·재전송 프로토타입을 단계별로 구현한다. 최종 목표는 대역폭이 제한된 상황에서도 FLASH·IMMEDIATE 전문을 기한 안에 전달하고, 어떤 전문도 잃어버리지 않으며, 순서 조정의 최종 판단은 통신관이 내리도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c202_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c202.py` | 전처리, 우선순위 큐, 재전송 제어, 링크 신뢰도, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c202_data_spec.md`, `docs/c202_hitl_guide.md`, `docs/c202_aar_report.md`, `docs/c202_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c202_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c202_stress_test.py` | 대역폭 90% 급감 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c202_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 최우선 순위 전문 강제 전송 및 큐 순서 수동 조정을 위한 통신관 HITL 통제 기능 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

최우선 작전 전문 강제 전송 및 큐 순서 수동 조정을 위한 통신관 HITL 제어 기능을 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **통신관이 최우선 전문 강제 전송과 큐 순서 수동 조정을 직접 수행하는 HITL 통제 기능을 설계한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **시스템은 보류·앞당기기를 추천만 하고 큐 순서를 바꾸지 않는다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/hitl/c202_approval_workflow.py 및 docs/c202_hitl_guide.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 지휘 책임 분계선 명확화 및 성급한 자동 단정 방지 체계 부합성

#### 미션 PASS 조건

- 정량 기준: 인간 검토 큐(Queue) 대기시간 및 승인/반려 처리 로깅 성공률 100%


- 진행 기준: 승인/반려 인터페이스 동작 및 감사 로그(Audit Log) 생성 검증


- 검증 방법: 모의 조작자 승인/반려 시뮬레이션 및 데이터 무결성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""통신관이 최우선 전문 강제 전송과 큐 순서 수동 조정을 직접 수행하는 HITL 통제 기능을 구현한다. 시스템은 추천만 하고 순서를 스스로 바꾸지 않으며, 규칙 위반 조작은 차단하고, FLASH보다 앞서는 강제 전송은 상급자 승인을 거치게 하며, 모든 조작은 위·변조 확인이 가능한 감사 로그에 기록한다."""
import hashlib
import json
import pandas as pd

ACTIONS = {"FORCE_SEND": "강제 전송(맨 앞)", "MOVE_UP": "한 칸 앞으로", "MOVE_DOWN": "한 칸 뒤로", "HOLD": "보류", "RELEASE": "보류 해제"}
RANK = {"FLASH": 0, "IMMEDIATE": 1, "PRIORITY": 2, "ROUTINE": 3}


class ControlError(ValueError):
    pass


class FakeClock:
    t = 1000.0

    def __call__(self):
        return self.t


class HITLQueue:
    def __init__(self, clock):
        self.q, self.held, self.req, self.log, self.clock = [], {}, {}, [], clock

    def _record(self, body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {**body, "ts": round(self.clock(), 1)}
        h = hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest()
        self.log.append({**body, "prev": prev, "hash": h})

    def verify_log(self):
        prev = "GENESIS"
        for n, rec in enumerate(self.log, start=1):
            body = {k: v for k, v in rec.items() if k not in ("prev", "hash")}
            if rec["prev"] != prev or rec["hash"] != hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest():
                return False, n
            prev = rec["hash"]
        return True, -1

    def submit(self, mid, prec):
        """기본 정렬(긴급도 -> 도착순) 위치에 넣는다. 수동 조정된 순서는 보존한다."""
        m = {"id": mid, "prec": prec, "t0": self.clock()}
        pos = next((i for i, x in enumerate(self.q) if RANK[x["prec"]] > RANK[prec]), len(self.q))
        self.q.insert(pos, m)
        self._record({"event": "SUBMIT", "id": mid, "prec": prec, "by": "SYSTEM"})

    def recommend(self, link_grade):
        """시스템 추천(자동 적용 안 함): 링크 Low면 ROUTINE 대용량 보류 권고 등."""
        out = []
        if link_grade == "Low":
            out += [f"{m['id']} 보류 권고(ROUTINE)" for m in self.q if m["prec"] == "ROUTINE"][:2]
        old = [m for m in self.q if m["prec"] == "IMMEDIATE" and self.clock() - m["t0"] > 20]
        out += [f"{m['id']} 앞당기기 권고(기한 임박)" for m in old]
        self._record({"event": "RECOMMEND", "items": out, "by": "SYSTEM"})
        return out

    def act(self, mid, action, operator, reason, approver=None):
        if action not in ACTIONS:
            raise ControlError(f"허용되지 않은 조작: {action}")
        if not operator:
            raise ControlError("통신관 ID 필요")
        if len(reason.strip()) < 5:
            raise ControlError("조작 근거 5자 이상 필요")
        idx = next((i for i, m in enumerate(self.q) if m["id"] == mid), None)
        if idx is None and not (action == "RELEASE" and mid in self.held):
            raise ControlError("큐에 없는 전문")
        if action == "FORCE_SEND":
            m = self.q[idx]
            flash_waiting = any(x["prec"] == "FLASH" for x in self.q if x["id"] != mid)
            if m["prec"] != "FLASH" and flash_waiting and not approver:
                self.req[mid] = {"operator": operator, "reason": reason, "t": self.clock()}
                self._record({"event": "REQUEST", "id": mid, "action": action, "operator": operator, "reason": reason})
                return "상급자 승인 대기(대기 중인 FLASH보다 앞섬)"
            if approver == operator:
                raise ControlError("요청자와 승인자는 달라야 함(2인 확인)")
            self.q.insert(0, self.q.pop(idx)); self.req.pop(mid, None)
        elif action in ("MOVE_UP", "MOVE_DOWN"):
            j = idx - 1 if action == "MOVE_UP" else idx + 1
            if not 0 <= j < len(self.q):
                raise ControlError("이동할 위치 없음")
            if action == "MOVE_UP" and self.q[j]["prec"] == "FLASH" and self.q[idx]["prec"] != "FLASH" and not approver:
                raise ControlError("FLASH 앞으로 이동은 상급자 승인 필요")
            self.q[idx], self.q[j] = self.q[j], self.q[idx]
        elif action == "HOLD":
            if self.q[idx]["prec"] == "FLASH" and (len(reason.strip()) < 15 or not approver):
                raise ControlError("FLASH 보류는 근거 15자 이상 + 상급자 승인 필요")
            self.held[mid] = self.q.pop(idx)
        elif action == "RELEASE":
            m = self.held.pop(mid)
            pos = next((i for i, x in enumerate(self.q) if RANK[x["prec"]] > RANK[m["prec"]]), len(self.q))
            self.q.insert(pos, m)
        self._record({"event": "ACT", "id": mid, "action": action, "operator": operator, "approver": approver, "reason": reason})
        return "적용"


clock = FakeClock()
hq = HITLQueue(clock)
# [교육용가상자료] 대기 전문 7건
for mid, p in [("M0101", "ROUTINE"), ("M0102", "PRIORITY"), ("M0103", "FLASH"), ("M0104", "IMMEDIATE"), ("M0105", "ROUTINE"), ("M0106", "PRIORITY"), ("M0107", "IMMEDIATE")]:
    clock.t += 3; hq.submit(mid, p)
order0 = [m["id"] for m in hq.q]
print("초기 큐 순서:", order0)
clock.t += 20
print("시스템 추천(자동 적용 안 함):", hq.recommend("Low"))
print("추천 후 큐 순서 변화 없음:", [m["id"] for m in hq.q] == order0)

# 규칙 위반 조작 차단
for args in [("M0105", "FORCE_SEND", "", "근거 있음"), ("M0102", "MOVE_UP", "sig01", "앞으로"), ("M0104", "MOVE_UP", "sig01", "FLASH 앞으로 이동 요청"),
             ("M0103", "HOLD", "sig01", "보류 필요함"), ("M0101", "DELETE", "sig01", "삭제 요청함")]:
    try:
        hq.act(*args)
    except ControlError as e:
        print(f"차단: {args[0]} {args[1]} -> {e}")

# 정상 조작
clock.t += 2; print("M0107 앞당김:", hq.act("M0107", "MOVE_UP", "sig01", "기한 임박 IMMEDIATE 우선 처리"))
clock.t += 2; print("M0102 강제 전송:", hq.act("M0102", "FORCE_SEND", "sig01", "지휘관 지시 상황보고 즉시 전송"))
clock.t += 25; print("M0102 상급자 승인:", hq.act("M0102", "FORCE_SEND", "sig01", "지휘관 지시 상황보고 즉시 전송", approver="sup01"))
clock.t += 2; print("M0105 보류:", hq.act("M0105", "HOLD", "sig02", "링크 Low, 대용량 행정전문 보류"))
print("조정 후 큐 순서:", [m["id"] for m in hq.q], "/ 보류:", list(hq.held))

# 대기시간·로그
waits = [{"id": r["id"], "조작": r["action"], "통신관": r["operator"], "승인자": r["approver"] or "-",
          "요청 후 적용까지(s)": round(r["ts"] - next((x["ts"] for x in hq.log if x["event"] == "REQUEST" and x["id"] == r["id"]), r["ts"]), 1)}
         for r in hq.log if r["event"] == "ACT"]
print("\n" + pd.DataFrame(waits).to_string(index=False))
n_act = sum(r["event"] in ("ACT", "REQUEST") for r in hq.log)
print(f"\n감사 로그 {len(hq.log)}건(조작·요청 {n_act}건 전부 기록, 로깅 성공률 100%), 해시 체인 검증: {hq.verify_log()}")
hq.log[9]["operator"] = "sig99"
print("10번째 기록 변조 후 검증(False, 깨진 위치):", hq.verify_log())

# 지휘관 피드백 입력 양식(문자열로 출력)
form = {"form": "c202_commander_feedback", "date_time": "(입력)", "message_id": "M0102", "delivery_ok": "(Y/N)",
        "delay_acceptable": "(Y/N)", "priority_change_request": "(내용)", "comment": "(내용)"}
print("\n[지휘관 피드백 입력 양식]\n" + json.dumps(form, ensure_ascii=False))
print("시스템은 순서를 추천만 하며, 강제 전송·보류의 책임은 조작한 통신관과 승인한 상급자에게 있다.")


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
초기 큐 순서: ['M0103', 'M0104', 'M0107', 'M0102', 'M0106', 'M0101', 'M0105']
시스템 추천(자동 적용 안 함): ['M0101 보류 권고(ROUTINE)', 'M0105 보류 권고(ROUTINE)', 'M0104 앞당기기 권고(기한 임박)']
추천 후 큐 순서 변화 없음: True
차단: M0105 FORCE_SEND -> 통신관 ID 필요
차단: M0102 MOVE_UP -> 조작 근거 5자 이상 필요
차단: M0104 MOVE_UP -> FLASH 앞으로 이동은 상급자 승인 필요
차단: M0103 HOLD -> FLASH 보류는 근거 15자 이상 + 상급자 승인 필요
차단: M0101 DELETE -> 허용되지 않은 조작: DELETE
M0107 앞당김: 적용
M0102 강제 전송: 상급자 승인 대기(대기 중인 FLASH보다 앞섬)
M0102 상급자 승인: 적용
M0105 보류: 적용
조정 후 큐 순서: ['M0102', 'M0103', 'M0107', 'M0104', 'M0106', 'M0101'] / 보류: ['M0105']

   id         조작   통신관   승인자  요청 후 적용까지(s)
M0107    MOVE_UP sig01     -           0.0
M0102 FORCE_SEND sig01 sup01          25.0
M0105       HOLD sig02     -           0.0

감사 로그 12건(조작·요청 4건 전부 기록, 로깅 성공률 100%), 해시 체인 검증: (True, -1)
10번째 기록 변조 후 검증(False, 깨진 위치): (False, 10)

[지휘관 피드백 입력 양식]
{"form": "c202_commander_feedback", "date_time": "(입력)", "message_id": "M0102", "delivery_ok": "(Y/N)", "delay_acceptable": "(Y/N)", "priority_change_request": "(내용)", "comment": "(내용)"}
시스템은 순서를 추천만 하며, 강제 전송·보류의 책임은 조작한 통신관과 승인한 상급자에게 있다.
```
</details>
